# Chapter 10: Web Application Security

> "All input is evil until proven otherwise." Michael Howard and David LeBlanc

---

## Learning Objectives

After completing this chapter, you will be able to:

1. Describe the OWASP Top 10 categories of web application risk.
2. Explain injection, broken access control, and cross-site scripting.
3. Apply input validation, output encoding, and parameterization as defenses.
4. Reason about authentication and session management weaknesses.
5. Recognize secure-by-default patterns in code.

## Key Terms

- **OWASP**: Open Worldwide Application Security Project.
- **XSS**: Cross-Site Scripting.
- **CSRF**: Cross-Site Request Forgery.
- **SQLi**: SQL Injection.
- **WAF**: Web Application Firewall.

---

## 10.1 The OWASP Top 10

The OWASP Top 10 is a community-driven awareness document ranking the most critical web application
security risks {cite}`owasp_top10_2021`. Recent editions emphasize broken access control, cryptographic
failures, and injection, among others. It is not a complete checklist, but it is an excellent map of
where defensive effort tends to pay off and a common vocabulary across the industry.

## 10.2 Injection

Injection occurs when untrusted input is interpreted as code or commands. SQL injection, in which input
alters a database query, remains a canonical example and is cataloged as CWE-89 {cite}`cwe`. The
durable fix is to separate code from data using parameterized queries or prepared statements, so that
user input is always treated as a value and never as executable syntax. Output encoding and least
privilege provide defense in depth.

## 10.3 Broken Access Control

Access control enforces what an authenticated user is allowed to do. It breaks when the application
trusts client-supplied identifiers, fails to check authorization on every request, or exposes functions
by obscurity alone. The result can be one user reading or modifying another user's data. Robust designs
check authorization server-side for every sensitive action against the authenticated identity.

## 10.4 Cross-Site Scripting

Cross-site scripting injects script into pages viewed by other users, letting an attacker run code in
the victim's browser context to steal sessions or perform actions as the victim. Defenses include
encoding output for the correct context, validating input, and using a content security policy. The
related cross-site request forgery tricks a browser into sending authenticated requests, mitigated with
anti-forgery tokens and same-site cookies.

## 10.5 Why This Matters

Web applications are the most exposed software most organizations run, reachable by anyone with a
browser. A single injection or access-control flaw can expose an entire database. The defensive
patterns here are inexpensive when designed in and very costly when retrofitted after a breach.

## 10.6 News in Focus

The 2017 breach mentioned earlier in this book began with the exploitation of a known vulnerability in a
widely used web application framework that had not been patched. It is a recurring lesson that web
security depends as much on disciplined dependency and patch management as on writing careful code.

## 10.7 Worked Example: Safe vs Unsafe Query Construction

The cell contrasts string concatenation, which enables injection, with parameterized queries, which do
not. It uses an in-memory SQLite database so it runs anywhere.


In [1]:
import sqlite3

conn = sqlite3.connect(":memory:")
c = conn.cursor()
c.execute("CREATE TABLE users(id INTEGER, name TEXT, secret TEXT)")
c.executemany("INSERT INTO users VALUES (?,?,?)",
              [(1,"alice","alpha"),(2,"bob","bravo")])
conn.commit()

malicious = "alice' OR '1'='1"

# UNSAFE: concatenation lets the input change the query logic
unsafe_sql = "SELECT name, secret FROM users WHERE name = '" + malicious + "'"
print("Unsafe query text:")
print(" ", unsafe_sql)
print("  Rows returned (leaks every user):", c.execute(unsafe_sql).fetchall())

# SAFE: parameterization treats input strictly as a value
print("\nSafe parameterized query:")
print("  Rows returned:", c.execute(
    "SELECT name, secret FROM users WHERE name = ?", (malicious,)).fetchall())
print("  The malicious input matched no user, exactly as intended.")
conn.close()


Unsafe query text:
  SELECT name, secret FROM users WHERE name = 'alice' OR '1'='1'
  Rows returned (leaks every user): [('alice', 'alpha'), ('bob', 'bravo')]

Safe parameterized query:
  Rows returned: []
  The malicious input matched no user, exactly as intended.


## 10.8 Review Questions (MCQ)

**Q1.** The durable fix for SQL injection is:
A. Input length limits  B. Parameterized queries  C. Hiding errors  D. A faster database

**Q2.** XSS executes in the context of the:
A. Database server  B. Web server  C. Victim's browser  D. Firewall

**Q3.** Anti-forgery tokens primarily defend against:
A. SQLi  B. XSS  C. CSRF  D. DoS

*Answers: Q1 B, Q2 C, Q3 C.*

## 10.9 Lab Assignment

Deploy a deliberately vulnerable web application in an isolated lab. Demonstrate one injection and one
access-control issue, then apply the corresponding fix and show that the attack no longer works.
Document before and after behavior.

## References

```{bibliography}
:filter: docname in docnames
```
